# Notebook 11: Streams — Event Streaming & Message Queues

Redis Streams (added in Redis 5.0) are an **append-only log** data structure. Think of them as a combination of:
- **Pub/Sub** — real-time message delivery
- **Lists** — persistent, ordered messages
- **Kafka** — consumer groups, acknowledgment, replay

```
Comparison:
  Pub/Sub  = Live radio     (miss it, it's gone)
  Lists    = Mailbox        (one reader per message)
  Streams  = DVR recording  (replay, multiple readers, acknowledgment)
```

Streams solve all of Pub/Sub's limitations: persistence, replay, consumer groups, acknowledgment.

In [ ]:
import redis
import time

r = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)
r.flushdb()
print("Connected and ready!")

---
## 1. XADD — Adding Entries to a Stream

Each stream entry has:
- An **ID** (auto-generated as `<timestamp>-<sequence>`)
- **Field-value pairs** (like a mini hash)

In [ ]:
# Redis CLI: XADD mystream * sensor temp value 22.5
# The '*' means auto-generate the ID

id1 = r.xadd('sensor:data', {'sensor': 'temperature', 'value': '22.5', 'unit': 'celsius'})
print(f"Entry 1 ID: {id1}")

time.sleep(0.01)  # Small delay so timestamps differ

id2 = r.xadd('sensor:data', {'sensor': 'humidity', 'value': '65', 'unit': 'percent'})
print(f"Entry 2 ID: {id2}")

id3 = r.xadd('sensor:data', {'sensor': 'temperature', 'value': '23.1', 'unit': 'celsius'})
print(f"Entry 3 ID: {id3}")

print(f"\nID format: <timestamp_ms>-<sequence>")
print(f"Stream length: {r.xlen('sensor:data')}")

## 2. XRANGE / XREVRANGE — Reading Entries

In [ ]:
# XRANGE — Read entries (oldest to newest)
# '-' means minimum ID, '+' means maximum ID
# Redis CLI: XRANGE sensor:data - +

entries = r.xrange('sensor:data', '-', '+')
print("All entries (oldest first):")
for entry_id, fields in entries:
    print(f"  [{entry_id}] {fields}")

# Get only the last 2 entries
print("\nLast 2 entries (newest first):")
entries = r.xrevrange('sensor:data', '+', '-', count=2)
for entry_id, fields in entries:
    print(f"  [{entry_id}] {fields}")

## 3. XREAD — Read New Entries (Streaming)

XREAD is like a cursor — you read from a specific ID onwards.

In [ ]:
# Read all entries from the beginning ('0' = start)
result = r.xread({'sensor:data': '0'}, count=10)
print("XREAD from beginning:")
for stream_name, entries in result:
    print(f"  Stream: {stream_name}")
    for entry_id, fields in entries:
        print(f"    [{entry_id}] {fields['sensor']}: {fields['value']}")

# Read only NEW entries (after the last ID we saw)
# Add more entries first
r.xadd('sensor:data', {'sensor': 'pressure', 'value': '1013', 'unit': 'hPa'})

# Read only entries after id3
result = r.xread({'sensor:data': id3}, count=10)
print(f"\nNew entries after {id3}:")
for stream_name, entries in result:
    for entry_id, fields in entries:
        print(f"  [{entry_id}] {fields}")

In [ ]:
# XREAD with blocking — wait for new entries
import threading

def stream_producer():
    time.sleep(1)
    producer = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)
    new_id = producer.xadd('sensor:data', {'sensor': 'light', 'value': '450', 'unit': 'lux'})
    print(f"  [Producer] Added entry: {new_id}")

# Start producer that will add data after 1 second
t = threading.Thread(target=stream_producer, daemon=True)
t.start()

# Block-read, waiting for new data ($ means only new entries)
print("Waiting for new stream entry (blocking)...")
result = r.xread({'sensor:data': '$'}, block=3000, count=1)  # Wait up to 3 seconds

if result:
    for stream_name, entries in result:
        for entry_id, fields in entries:
            print(f"  [Consumer] Got: {fields}")
else:
    print("  Timed out!")

t.join(timeout=2)

---
## 4. Consumer Groups — The Killer Feature

Consumer groups let **multiple consumers** process a stream **without duplicating work**.

```
Stream: [msg1, msg2, msg3, msg4, msg5, msg6]

Consumer Group "workers":
  Consumer A gets: msg1, msg3, msg5
  Consumer B gets: msg2, msg4, msg6
  (each message delivered to exactly ONE consumer)
```

This is how you build scalable message processing!

In [ ]:
r.flushdb()

# Create a stream with some data
for i in range(6):
    r.xadd('orders', {'order_id': str(i+1), 'product': f'item_{i+1}', 'amount': str((i+1) * 10)})

print(f"Stream has {r.xlen('orders')} entries")

# Create a consumer group starting from the beginning ('0')
# Redis CLI: XGROUP CREATE orders processors 0
r.xgroup_create('orders', 'processors', '0')
print("Consumer group 'processors' created!")

In [ ]:
# Consumer A reads from the group
# '>' means "give me new messages that no one in this group has seen"
# Redis CLI: XREADGROUP GROUP processors worker_A COUNT 3 STREAMS orders >

msgs_a = r.xreadgroup('processors', 'worker_A', {'orders': '>'}, count=3)
print("Worker A got:")
for stream, entries in msgs_a:
    for entry_id, fields in entries:
        print(f"  [{entry_id}] Order #{fields['order_id']}: {fields['product']}")

# Consumer B reads the NEXT batch
msgs_b = r.xreadgroup('processors', 'worker_B', {'orders': '>'}, count=3)
print("\nWorker B got:")
for stream, entries in msgs_b:
    for entry_id, fields in entries:
        print(f"  [{entry_id}] Order #{fields['order_id']}: {fields['product']}")

print("\nEach order was delivered to exactly ONE worker!")

### XACK — Acknowledging Processed Messages

After a consumer processes a message, it should **acknowledge** it. Unacknowledged messages can be **reclaimed** by other consumers if the original consumer crashes.

In [ ]:
# Check pending (unacknowledged) messages
pending = r.xpending('orders', 'processors')
print(f"Pending messages: {pending}")

# Acknowledge worker_A's messages
for stream, entries in msgs_a:
    for entry_id, fields in entries:
        r.xack('orders', 'processors', entry_id)
        print(f"  ACK'd: {entry_id}")

# Check pending again
pending = r.xpending('orders', 'processors')
print(f"\nPending after ACK: {pending}")
print("Worker A's messages are acknowledged, Worker B's are still pending.")

In [ ]:
# ACK worker_B's messages too
for stream, entries in msgs_b:
    for entry_id, fields in entries:
        r.xack('orders', 'processors', entry_id)

pending = r.xpending('orders', 'processors')
print(f"All pending cleared: {pending}")

## 5. XTRIM — Limit Stream Size

In [ ]:
# Add many entries
for i in range(20):
    r.xadd('logs', {'level': 'info', 'msg': f'Log message {i}'})

print(f"Before trim: {r.xlen('logs')} entries")

# Keep only the last 10 entries
# Redis CLI: XTRIM logs MAXLEN 10
r.xtrim('logs', maxlen=10)
print(f"After trim:  {r.xlen('logs')} entries")

# You can also use approximate trimming (more efficient)
# Redis CLI: XTRIM logs MAXLEN ~ 10
# The ~ means "approximately 10" — Redis may keep a few extra

## 6. XINFO — Stream Information

In [ ]:
# Stream info
info = r.xinfo_stream('orders')
print("Stream info:")
print(f"  Length: {info['length']}")
print(f"  First entry: {info['first-entry']}")
print(f"  Last entry: {info['last-entry']}")

# Consumer group info
groups = r.xinfo_groups('orders')
print(f"\nConsumer groups:")
for group in groups:
    print(f"  Group: {group['name']}")
    print(f"    Consumers: {group['consumers']}")
    print(f"    Pending: {group['pending']}")

---
## 7. Real-World: Event Log System

In [ ]:
import json

r.flushdb()

def emit_event(event_type, data):
    """Emit an event to the stream."""
    entry = {'type': event_type, 'data': json.dumps(data), 'timestamp': str(time.time())}
    return r.xadd('events', entry)

def get_recent_events(count=10):
    """Get the N most recent events."""
    entries = r.xrevrange('events', '+', '-', count=count)
    events = []
    for entry_id, fields in entries:
        events.append({
            'id': entry_id,
            'type': fields['type'],
            'data': json.loads(fields['data'])
        })
    return events

# Emit various events
emit_event('user.signup', {'user_id': 1001, 'name': 'Sujit'})
emit_event('order.placed', {'order_id': 42, 'user_id': 1001, 'total': 99.99})
emit_event('payment.received', {'order_id': 42, 'method': 'card'})
emit_event('order.shipped', {'order_id': 42, 'tracking': 'TRK123'})
emit_event('user.login', {'user_id': 1001, 'ip': '192.168.1.1'})

# View recent events
print("Recent events:")
for event in get_recent_events(5):
    print(f"  [{event['type']}] {event['data']}")

---
## 8. Real-World: Order Processing Pipeline

In [ ]:
r.flushdb()

# Create the stream and consumer groups for each stage
r.xadd('order_pipeline', {'init': 'true'})  # Initialize stream
r.xgroup_create('order_pipeline', 'payment_team', '0')
r.xgroup_create('order_pipeline', 'shipping_team', '0')

# Place some orders
orders = [
    {'order_id': '101', 'item': 'Laptop', 'amount': '999'},
    {'order_id': '102', 'item': 'Mouse', 'amount': '25'},
    {'order_id': '103', 'item': 'Keyboard', 'amount': '75'},
]

for order in orders:
    r.xadd('order_pipeline', order)
    print(f"  Order placed: #{order['order_id']} - {order['item']}")

print("\n--- Payment Team Processing ---")
result = r.xreadgroup('payment_team', 'payment_worker_1', {'order_pipeline': '>'}, count=10)
for stream, entries in result:
    for entry_id, fields in entries:
        if 'order_id' in fields:
            print(f"  Processing payment for order #{fields['order_id']}: ${fields['amount']}")
            r.xack('order_pipeline', 'payment_team', entry_id)

print("\n--- Shipping Team Processing ---")
result = r.xreadgroup('shipping_team', 'shipping_worker_1', {'order_pipeline': '>'}, count=10)
for stream, entries in result:
    for entry_id, fields in entries:
        if 'order_id' in fields:
            print(f"  Shipping order #{fields['order_id']}: {fields['item']}")
            r.xack('order_pipeline', 'shipping_team', entry_id)

print("\nBoth teams processed the same orders independently!")

---
## Streams vs Pub/Sub vs Lists

| Feature | Pub/Sub | Lists | Streams |
|---|---|---|---|
| Persistent | No | Yes | Yes |
| Replay history | No | No | Yes |
| Consumer groups | No | No | Yes |
| Acknowledgment | No | No | Yes (XACK) |
| Multiple consumers | All get same | One gets each | Configurable |
| Blocking read | Yes | Yes (BLPOP) | Yes (XREAD BLOCK) |
| ID/ordering | No | By position | By ID (timestamp) |
| Best for | Broadcast | Simple queue | Robust messaging |

---
## Cleanup

In [ ]:
r.flushdb()
print("Cleaned up!")

---
## Key Takeaways

```
XADD stream * field value       → Add entry (auto ID)
XLEN stream                     → Count entries
XRANGE stream - +               → Read all (oldest first)
XREVRANGE stream + -            → Read all (newest first)
XREAD COUNT n STREAMS stream 0  → Read from position
XREAD BLOCK ms STREAMS stream $ → Block-read new entries
XTRIM stream MAXLEN n           → Limit stream size

# Consumer Groups
XGROUP CREATE stream group 0         → Create group
XREADGROUP GROUP g consumer STREAMS s > → Read as consumer
XACK stream group id                  → Acknowledge message
XPENDING stream group                 → Check pending
XINFO STREAM stream                   → Stream info
XINFO GROUPS stream                   → Group info
```

---
## Exercises

1. **Chat with History:** Build a chat system using Streams instead of Pub/Sub. New users should be able to see the last 50 messages when they join.

2. **Worker Pool:** Create a stream with 20 tasks. Set up 3 consumers in a group. Have each consumer process tasks and acknowledge them. Verify all tasks are processed exactly once.

3. **Dead Letter Queue:** Implement a pattern where messages pending for more than 10 seconds are moved to a "dead letter" stream using XCLAIM.

4. **Real-Time Dashboard:** Create a stream of sensor data. Write a consumer that calculates a running average of the last 10 readings.

In [ ]:
# Your exercises here!


---
**Next up: [Notebook 12 — Lua Scripting](./12_Lua_Scripting.ipynb)** — Custom atomic operations by running scripts directly on the Redis server!